# Inventory Forecasting
**Ecommerce AI Series - Nub Labs Playbooks**

Predict monthly demand per product category from order history.\
Compare a rolling mean baseline against Prophet time series forecasting.\
Output: reorder point recommendations per category.

Dataset: [TechHeaven Reference Platform](https://github.com/Nub-Labs/techheaven-reference-platform) -
5,000 orders across 15 categories (Jan 2025 - Jun 2026). No API key required.

## 1. Business problem

A retailer buys inventory based on what sold last month. Last month is not a forecast - it is history. It does not tell you that laptop sales will spike 78% in October when back-to-school and pre-holiday buying peaks. It does not tell you that Smart Home devices trend upward in Q4. It does not tell you that the product you ordered 50 units of in September will sell out in the first two weeks of October.

The data to predict this already exists in the order history. Every sale is a timestamped signal. A demand forecasting model reads those signals, extracts the trend, identifies the seasonal pattern, and produces a prediction with confidence intervals: here is how many units category X will likely sell next month, and here is the reorder quantity that keeps you in stock with 95% confidence.

TechHeaven has 5,000 orders across 15 categories spanning 18 months (January 2025 - June 2026). The Q4 spike in October-November 2025 is the central test case. A rolling mean cannot predict it. Prophet, which models trend and seasonality explicitly, learns from it and forecasts the next cycle.

In [ ]:
# Prophet install takes 2-3 minutes on a fresh Colab runtime
import sys
if 'google.colab' in sys.modules:
    !pip install prophet scikit-learn pandas numpy matplotlib seaborn -q

## 2. The dataset

Two files from the TechHeaven public repository:

- `transactions/orders.json` - 5,000 orders: order_id, customer_id, status, order_date
- `transactions/order_items.json` - 15,000 line items: order_id, product_id, product_name, category, qty, price

Joining on `order_id` and filtering to `completed` and `processing` orders produces 4,250 confirmed orders. Aggregating by month and category gives the time series each model trains on.

Monthly aggregation is used throughout this notebook. At the product level, TechHeaven has ~17 orders per product - too sparse for time series modelling. At the category level, 15 categories with an average of 283 orders each produce 18 monthly observations - sufficient for trend and seasonal pattern detection.

In [ ]:
import json
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from prophet import Prophet

BASE = 'https://raw.githubusercontent.com/Nub-Labs/techheaven-reference-platform/main/data'

ROLLING_WINDOW   = 3    # months
LEAD_TIME_MONTHS = 1    # assumed supplier lead time for reorder calculation
FORECAST_MONTHS  = 6    # forward forecast horizon

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

In [ ]:
def fetch_json(url: str) -> list:
    with urllib.request.urlopen(url) as r:
        return json.load(r)

orders_raw = fetch_json(f'{BASE}/transactions/orders.json')
items_raw  = fetch_json(f'{BASE}/transactions/order_items.json')

orders_df = pd.DataFrame(orders_raw)
items_df  = pd.DataFrame(items_raw)

print(f'Orders:      {len(orders_df):,}')
print(f'Order items: {len(items_df):,}')
print(f'Categories:  {items_df["category"].nunique()}')

In [ ]:
df = items_df.merge(
    orders_df[['order_id', 'status', 'order_date']],
    on='order_id'
)
df = df[df['status'].isin(['completed', 'processing'])].copy()
df['order_date'] = pd.to_datetime(df['order_date'])

print(f'Date range:   {df["order_date"].min().date()} to {df["order_date"].max().date()}')
print(f'Orders kept:  {df["order_id"].nunique():,}  (excluded: cancelled and pending)')
print(f'Units sold:   {df["qty"].sum():,}')
print()
print('Units by category:')
print(df.groupby('category')['qty'].sum().sort_values(ascending=False).to_string())

In [ ]:
# All-category monthly demand - the Q4 spike is the central story
monthly_total = (
    df.groupby(df['order_date'].dt.to_period('M'))['qty']
    .sum()
    .reset_index()
)
monthly_total['order_date'] = monthly_total['order_date'].dt.to_timestamp()

is_q4 = monthly_total['order_date'].dt.month.isin([10, 11]) & (monthly_total['order_date'].dt.year == 2025)
colors = ['#ef4444' if q else '#6366f1' for q in is_q4]

baseline = monthly_total.loc[~is_q4, 'qty'].mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(monthly_total['order_date'], monthly_total['qty'], width=22, color=colors, alpha=0.85)
ax.axhline(baseline, color='#94a3b8', linestyle='--', linewidth=1.2)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#ef4444', alpha=0.85, label='Q4 spike (Oct-Nov 2025)'),
    Patch(facecolor='#6366f1', alpha=0.85, label='Normal months'),
    plt.Line2D([0],[0], color='#94a3b8', linestyle='--', label=f'Baseline avg ({baseline:.0f} units/month)'),
])
ax.set_title('TechHeaven monthly units sold across all categories', fontsize=13, fontweight='bold', pad=10)
ax.set_ylabel('Units sold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('monthly_demand_total.png', bbox_inches='tight')
plt.show()

for m in [10, 11]:
    row = monthly_total[monthly_total['order_date'].dt.month == m]
    v = row['qty'].values[0]
    print(f"{row['order_date'].dt.strftime('%B %Y').values[0]}: {v:,} units (+{(v/baseline-1)*100:.0f}% above baseline)")

## 3. Building the monthly demand series

Each model in this notebook trains on the same data: monthly units sold per category. The series has 18 observations per category (January 2025 to June 2026), with months that had no sales recorded as zero demand.

In [ ]:
# Monthly units sold per category - fill missing months with 0
df['month_start'] = df['order_date'].dt.to_period('M').dt.to_timestamp()

raw_monthly = (
    df.groupby(['month_start', 'category'])['qty']
    .sum()
    .reset_index()
    .rename(columns={'month_start': 'ds', 'qty': 'y'})
)

# Months with no sales are true zeros, not missing data - fill the full 18-month grid
all_dates  = pd.period_range(
    df['order_date'].dt.to_period('M').min(),
    df['order_date'].dt.to_period('M').max(),
    freq='M'
).to_timestamp()
categories = sorted(raw_monthly['category'].unique())

idx = pd.MultiIndex.from_product([all_dates, categories], names=['ds', 'category'])
monthly = (
    raw_monthly.set_index(['ds', 'category'])
    .reindex(idx, fill_value=0)
    .reset_index()
)

print(f'Categories:          {len(categories)}')
print(f'Months per category: {monthly.groupby("category")["ds"].count().min()} (all equal to {len(all_dates)})')
print(f'Date range:          {monthly["ds"].min().date()} to {monthly["ds"].max().date()}')

In [ ]:
# Monthly demand for the 5 highest-volume categories
top5 = (
    monthly.groupby('category')['y'].sum()
    .sort_values(ascending=False)
    .head(5).index.tolist()
)

fig, axes = plt.subplots(5, 1, figsize=(13, 14), sharex=True)
palette = sns.color_palette('husl', 5)

for ax, cat, color in zip(axes, top5, palette):
    cat_data = monthly[monthly['category'] == cat].sort_values('ds')
    ax.fill_between(cat_data['ds'], cat_data['y'], alpha=0.2, color=color)
    ax.plot(cat_data['ds'], cat_data['y'], color=color, linewidth=2, marker='o', markersize=3)
    ax.set_ylabel('Units', fontsize=9)
    ax.set_title(cat, fontsize=10, fontweight='bold')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45, ha='right')
fig.suptitle('Monthly units sold - top 5 categories', fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig('monthly_by_category.png', bbox_inches='tight')
plt.show()

## 4. What a rolling mean cannot see

A rolling mean forecast predicts next month by averaging the previous N months. It is simple, fast, and used explicitly or implicitly by most retailers: "we sold about 300 units a month recently, so order 300."

The problem is structural: a rolling mean has no concept of seasonality. It cannot predict that October demand will be 78% above the August-September average. It learns from the recent past and projects it forward - a method that works well for stable demand and fails exactly when it matters most.

In [ ]:
# Rolling mean in-sample on Laptops - shows it lags the Q4 spike
cat = 'Laptops'
cat_data = monthly[monthly['category'] == cat].sort_values('ds').copy()

# One-step-ahead: predict month T using mean of months T-3 to T-1
cat_data['rolling_pred'] = cat_data['y'].shift(1).rolling(ROLLING_WINDOW).mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(cat_data['ds'], cat_data['y'], alpha=0.12, color='#6366f1')
ax.plot(cat_data['ds'], cat_data['y'],          color='#1e293b', linewidth=2, marker='o', markersize=4, label='Actual')
ax.plot(cat_data['ds'], cat_data['rolling_pred'], color='#f59e0b', linewidth=2, linestyle='--', marker='s', markersize=3, label=f'{ROLLING_WINDOW}-month rolling mean (1-step ahead)')

ax.set_title(f'{cat} - rolling mean lags behind the Q4 spike', fontsize=12, fontweight='bold', pad=10)
ax.set_ylabel('Units/month')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('rolling_mean_laptops.png', bbox_inches='tight')
plt.show()

In [ ]:
# Quantify the miss: rolling mean prediction vs actual for Oct and Nov 2025
cat = 'Laptops'
cat_data = monthly[monthly['category'] == cat].sort_values('ds').copy()
cat_data['rolling_pred'] = cat_data['y'].shift(1).rolling(ROLLING_WINDOW).mean()

q4_rows = cat_data[cat_data['ds'].dt.month.isin([10, 11]) & (cat_data['ds'].dt.year == 2025)]

print('Rolling mean: predicted vs actual (Laptops)')
print(f'{"Month":<15} {"Predicted":>12} {"Actual":>10} {"Miss":>10} {"Miss %":>8}')
print('-' * 58)
for _, row in q4_rows.iterrows():
    pred = row['rolling_pred']
    actual = row['y']
    miss = actual - pred
    pct = (miss / pred * 100) if pred > 0 else float('nan')
    print(f"{row['ds'].strftime('%B %Y'):<15} {pred:>12.0f} {actual:>10.0f} {miss:>10.0f} {pct:>7.0f}%")

## 5. Prophet learns the seasonal pattern

Prophet (Meta open-source) decomposes a time series into three additive components:

- **Trend** - long-term direction (growth, decline, plateau)
- **Seasonality** - repeating patterns at annual frequency
- **Residual** - unexplained variation

The critical advantage: the seasonality component is modelled explicitly as a Fourier series fitted to the historical data. After seeing one Q4 cycle (October-November 2025), Prophet encodes the annual peak in its seasonal component and projects it forward. It will predict the October 2026 spike even though October 2026 has not happened yet.

One Prophet model is fitted per category on the full 18-month dataset. The decomposition shows what the model learned. The forward forecast shows what it predicts next.

In [ ]:
def fit_prophet_monthly(cat_df: pd.DataFrame) -> Prophet:
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10.0,
    )
    m.fit(cat_df[['ds', 'y']])
    return m

models    = {}
forecasts = {}
latest_ds = monthly['ds'].max()

for cat in categories:
    cat_data = monthly[monthly['category'] == cat][['ds', 'y']].copy()
    m = fit_prophet_monthly(cat_data)
    future = m.make_future_dataframe(periods=FORECAST_MONTHS, freq='MS')
    fc = m.predict(future)
    models[cat]    = m
    forecasts[cat] = fc

print(f'Prophet models fitted: {len(models)} categories')
print(f'Forecast end date:     {forecasts[categories[0]]["ds"].max().date()}')

In [ ]:
# Prophet in-sample fit vs rolling mean vs actual - Laptops
cat = 'Laptops'
cat_data = monthly[monthly['category'] == cat].sort_values('ds').copy()
cat_data['rolling_pred'] = cat_data['y'].shift(1).rolling(ROLLING_WINDOW).mean()

fc = forecasts[cat]
insample = fc[fc['ds'] <= latest_ds]

fig, ax = plt.subplots(figsize=(13, 4.5))

ax.fill_between(insample['ds'], insample['yhat_lower'].clip(0), insample['yhat_upper'],
                alpha=0.15, color='#8b5cf6', label='Prophet 95% interval')

ax.plot(cat_data['ds'], cat_data['y'],            color='#1e293b', linewidth=2.2, marker='o', markersize=4, label='Actual', zorder=3)
ax.plot(insample['ds'], insample['yhat'].clip(0), color='#8b5cf6', linewidth=2,   linestyle='-',  label='Prophet (in-sample fit)')
ax.plot(cat_data['ds'], cat_data['rolling_pred'], color='#f59e0b', linewidth=1.8, linestyle='--', label=f'{ROLLING_WINDOW}-month rolling mean')

ax.set_title(f'{cat} - Prophet captures the Q4 spike, rolling mean misses it', fontsize=12, fontweight='bold', pad=10)
ax.set_ylabel('Units/month')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45, ha='right')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('prophet_vs_rolling_laptops.png', bbox_inches='tight')
plt.show()

In [ ]:
# Prophet decomposition for Laptops - annual seasonality shows the Q4 peak
cat = 'Laptops'
fig = models[cat].plot_components(forecasts[cat])
fig.set_size_inches(13, 7)
fig.suptitle(f'{cat} - Prophet decomposition: trend and annual seasonality', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('prophet_decomposition_laptops.png', bbox_inches='tight')
plt.show()
print('Yearly component: Q4 peak visible at month 10 - this is what rolling mean cannot encode.')

## 6. Forward forecast: predicting the next Q4

The 6-month forward forecast (July - December 2026) is the operational output. It shows what each category is expected to sell in the coming months - including the predicted Q4 spike in October-November 2026 that Prophet learned from the 2025 cycle.

A rolling mean predicts a flat line into the future. Prophet predicts the seasonal pattern. The difference matters most in August and September, when a buyer must decide whether to build inventory ahead of October demand.

In [ ]:
# Forward forecast for Laptops - shows Q4 2026 spike prediction
cat = 'Laptops'
cat_data = monthly[monthly['category'] == cat].sort_values('ds')
fc       = forecasts[cat]
future   = fc[fc['ds'] > latest_ds]
rolling_flat = cat_data['y'].tail(ROLLING_WINDOW).mean()

fig, ax = plt.subplots(figsize=(13, 4.5))

ax.fill_between(cat_data['ds'], cat_data['y'], alpha=0.12, color='#6366f1')
ax.plot(cat_data['ds'], cat_data['y'], color='#1e293b', linewidth=2, marker='o', markersize=3, label='Historical actual')

ax.fill_between(future['ds'], future['yhat_lower'].clip(0), future['yhat_upper'],
                alpha=0.2, color='#8b5cf6')
ax.plot(future['ds'], future['yhat'].clip(0), color='#8b5cf6', linewidth=2.2,
        linestyle='--', marker='D', markersize=4, label='Prophet forecast')

ax.plot(future['ds'], [rolling_flat] * len(future), color='#f59e0b', linewidth=1.8,
        linestyle='--', label=f'Rolling mean ({rolling_flat:.0f} units/month, flat)')

ax.axvline(latest_ds, color='#64748b', linestyle=':', linewidth=1.2, label='Forecast start')
ax.set_title(f'{cat} - Prophet predicts Q4 2026 spike, rolling mean projects flat', fontsize=12, fontweight='bold', pad=10)
ax.set_ylabel('Units/month')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45, ha='right')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('prophet_forward_laptops.png', bbox_inches='tight')
plt.show()

oct_pred = future[future['ds'].dt.month == 10]['yhat'].values
if len(oct_pred):
    print(f'Prophet October 2026 forecast: {oct_pred[0]:.0f} units (vs rolling mean flat: {rolling_flat:.0f})')
    print(f'Difference: +{oct_pred[0] - rolling_flat:.0f} units that rolling mean would miss')

In [ ]:
# Forward forecast for Smart Home
cat = 'Smart Home'
cat_data = monthly[monthly['category'] == cat].sort_values('ds')
fc       = forecasts[cat]
future   = fc[fc['ds'] > latest_ds]
rolling_flat = cat_data['y'].tail(ROLLING_WINDOW).mean()

fig, ax = plt.subplots(figsize=(13, 4))

ax.fill_between(cat_data['ds'], cat_data['y'], alpha=0.12, color='#10b981')
ax.plot(cat_data['ds'], cat_data['y'], color='#1e293b', linewidth=2, marker='o', markersize=3, label='Historical actual')
ax.fill_between(future['ds'], future['yhat_lower'].clip(0), future['yhat_upper'], alpha=0.2, color='#10b981')
ax.plot(future['ds'], future['yhat'].clip(0), color='#10b981', linewidth=2.2, linestyle='--', marker='D', markersize=4, label='Prophet forecast')
ax.plot(future['ds'], [rolling_flat] * len(future), color='#f59e0b', linewidth=1.8, linestyle='--', label=f'Rolling mean ({rolling_flat:.0f} units/month)')
ax.axvline(latest_ds, color='#64748b', linestyle=':', linewidth=1.2)

ax.set_title(f'{cat} - 6-month forward forecast', fontsize=12, fontweight='bold', pad=10)
ax.set_ylabel('Units/month')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45, ha='right')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('prophet_forward_smarthome.png', bbox_inches='tight')
plt.show()

## 7. Reorder point recommendations

The reorder quantity for each category is derived directly from the Prophet forecast:

1. **Forecast (next 30 days)** - Prophet point estimate for the coming month
2. **Upper bound (95%)** - the upper confidence interval; ordering to the upper bound provides a buffer against forecast uncertainty
3. **Safety stock** - one additional month's forecast demand at the point estimate, covering the assumed lead time
4. **Reorder quantity** = upper bound + safety stock

Using the upper confidence bound rather than the point estimate is deliberate: the cost of a stockout (lost sale, expedited reorder) exceeds the cost of a small overstock for most SKU categories.

In [ ]:
rows = []
for cat in categories:
    fc     = forecasts[cat]
    next_m = fc[fc['ds'] > latest_ds].head(1)

    forecast_units = max(0, next_m['yhat'].values[0])
    upper_units    = max(0, next_m['yhat_upper'].values[0])
    safety_stock   = max(0, forecast_units * LEAD_TIME_MONTHS)
    reorder_qty    = upper_units + safety_stock

    rows.append({
        'Category':          cat,
        'Forecast (units)':  round(forecast_units),
        'Upper bound (95%)': round(upper_units),
        'Safety stock':      round(safety_stock),
        'Reorder qty':       round(reorder_qty),
    })

reorder_df = (
    pd.DataFrame(rows)
    .sort_values('Reorder qty', ascending=False)
    .reset_index(drop=True)
)

print(f'Reorder recommendations - next 30 days from {latest_ds.date()}')
print(f'Lead time: {LEAD_TIME_MONTHS} month  |  Safety stock: {LEAD_TIME_MONTHS}x forecast demand')
print()
print(reorder_df.to_string(index=False))

## 8. Business interpretation

### The rolling mean vs Prophet divide

On a flat, stable series both models perform similarly. The difference emerges exactly when it matters most: ahead of a demand spike. A rolling mean trained on July-September 2025 data predicts flat demand for October. The actual October demand was 78% higher. By the time the rolling mean catches up - using the now-elevated October and November data - the spike is over.

Prophet encodes the Q4 seasonal pattern after observing it once. The forward forecast shows October 2026 predicted significantly above the summer baseline - not because October 2026 has been observed, but because the model learned the seasonal shape from October 2025. A buyer using Prophet starts building Laptop inventory in August. A buyer using rolling mean orders normally in August and scrambles to restock in October.

### The reorder table as an operational tool

The reorder table answers the question a buyer faces every month: how much do I order? The recommended quantity covers the upper confidence bound of next month's demand plus one month of safety stock against forecast error.

### What a production pipeline looks like

1. **Weekly batch job** - refresh Prophet models on the latest order data, regenerate the reorder table
2. **Alert integration** - trigger a purchase order when current stock falls below the reorder point
3. **Inventory system write-back** - push reorder quantities to Bagisto, Shopify, or a third-party WMS
4. **Monthly model review** - retrain as more Q4 cycles accumulate; the seasonal component improves with each observed year
5. **Product-level expansion** - once category-level is working, move to the top 20 SKUs by revenue

In [ ]:
# Key findings summary
laptops_oct = forecasts['Laptops'][forecasts['Laptops']['ds'].dt.month == 10]
laptops_oct_pred  = laptops_oct['yhat'].values
laptops_rolling   = monthly[monthly['category'] == 'Laptops']['y'].tail(ROLLING_WINDOW).mean()
top_cat = reorder_df.iloc[0]

print('Key findings')
print(f'  Q4 spike observed (Oct 2025):    +78% above baseline across all categories')
print(f'  Rolling mean Oct prediction:     {laptops_rolling:.0f} units/month (Laptops, flat)')
if len(laptops_oct_pred):
    delta = laptops_oct_pred[-1] - laptops_rolling
    print(f'  Prophet Oct 2026 forecast:       {laptops_oct_pred[-1]:.0f} units/month (Laptops)')
    print(f'  Inventory gap closed by Prophet: +{delta:.0f} units flagged 8 weeks early')
print()
print(f'  Top reorder category (next 30d): {top_cat["Category"]} - {top_cat["Reorder qty"]:.0f} units')
print()
print('Notebook:    github.com/Nub-Labs/playbooks')
print('Walkthrough: nublabs.com/playbooks/ecommerce/inventory-forecasting')